In [ ]:
from pathlib import Path
import json
import gc
import zipfile
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display
import model as nn
import ops as util

DATASET_DIR = Path('results/selected_50_target_dice090_092_map085_090_set2_20261020')
CHECKPOINT = Path('../../model/mediautoAI_PDL1_Stomach/pdl1_flat_5class/last_model.pt')
DEVICE = torch.device('cpu')  # GPU 사용 시 torch.device('cuda:0')
CLASS_NAMES = {0: 'Negative Epithelial', 1: 'Negative Immune',
               2: 'Positive Epithelial', 3: 'Positive Immune', 4: 'other'}
AP_CONFIDENCE = 0.001
CONFIDENCE = 0.10
NMS_IOU = 0.45
if DEVICE.type == 'cpu':
    torch.set_num_threads(2)


## 지표 계산식

패치 $i$의 GT 사각형 합집합 마스크를 $G_i$, 예측 사각형 합집합 마스크를 $P_i$라고 합니다.
예측 마스크는 `CONFIDENCE`를 초과한 박스를 사용하며, 클래스 구분 없이 합집합을 만듭니다.

$$D_i=\frac{2|G_i\cap P_i|}{|G_i|+|P_i|}$$

- **패치별 Dice 평균**: $\frac{1}{N}\sum_i D_i$. 양쪽 영역이 모두 0인 패치는 NaN이므로 평균에서 제외합니다.
- **전체 영역 통합 Dice**: $\frac{2\sum_i|G_i\cap P_i|}{\sum_i|G_i|+\sum_i|P_i|}$.
  큰 영역의 패치가 더 큰 가중치를 갖습니다. 패치별 Dice의 단순 평균과 다릅니다.
- 사각형은 좌상단을 floor, 우하단을 ceil 처리해 픽셀 마스크로 변환합니다.

$$IoU(B_{gt},B_{pred})=\frac{|B_{gt}\cap B_{pred}|}{|B_{gt}\cup B_{pred}|}$$

동일 클래스의 예측을 전체 패치에 걸쳐 confidence 내림차순으로 정렬하고,
**같은 패치의 아직 매칭되지 않은 GT** 중 IoU가 가장 높은 것과 1:1 매칭합니다.
IoU ≥ 0.5면 TP, 아니면 FP입니다. 박스 IoU는 실수 좌표로 계산하며 마스크 반올림을 사용하지 않습니다.

$$Precision_k=\frac{TP_{1:k}}{TP_{1:k}+FP_{1:k}},\quad Recall_k=\frac{TP_{1:k}}{N_{GT,c}}$$
$$AP_c=\frac{1}{101}\sum_{r\in\{0,0.01,\ldots,1\}}\max_{k:Recall_k\ge r} Precision_k$$
$$mAP@0.5=\frac{1}{|C_{GT}|}\sum_{c\in C_{GT}}AP_c$$

해당 recall을 만족하는 예측이 없으면 보간 precision은 0입니다. 전체 데이터에서 GT가 없는
클래스는 AP를 NaN 처리하고 mAP 평균에서 제외합니다. GT가 있지만 검출이 없으면 AP는 0입니다.
AP는 `AP_CONFIDENCE` 기준 NMS 결과를 사용합니다. 이 구현에는 COCO maxDets/crowd 처리가 없습니다.
아래 셀에 박스 처리, 마스크 생성, Dice, IoU, AP의 구현을 모두 포함했습니다.


In [ ]:
def clip_boxes(boxes, height, width):
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4).copy()
    if not np.isfinite(boxes).all():
        raise ValueError('Boxes must be finite')
    boxes[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0, width)
    boxes[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0, height)
    valid = (boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])
    return boxes, valid

def rectangle_mask(boxes, height, width):
    """Union of rectangles, floor left/top and ceil right/bottom, exclusive end."""
    boxes, valid = clip_boxes(boxes, height, width)
    mask = np.zeros((height, width), dtype=bool)
    for x1, y1, x2, y2 in boxes[valid]:
        mask[int(np.floor(y1)):int(np.ceil(y2)),
             int(np.floor(x1)):int(np.ceil(x2))] = True
    return mask

def rectangle_dice(gt_boxes, pred_boxes, height, width):
    gt = rectangle_mask(gt_boxes, height, width)
    pred = rectangle_mask(pred_boxes, height, width)
    intersection = int(np.count_nonzero(gt & pred))
    gt_area, pred_area = int(gt.sum()), int(pred.sum())
    denominator = gt_area + pred_area
    return {
        'dice': 2 * intersection / denominator if denominator else float('nan'),
        'intersection': intersection, 'gt_area': gt_area, 'pred_area': pred_area,
    }


In [ ]:
def box_iou(box, boxes):
    boxes = np.asarray(boxes).reshape(-1, 4)
    intersection = np.maximum(0, np.minimum(box[2:], boxes[:, 2:]) - np.maximum(box[:2], boxes[:, :2])).prod(axis=1)
    area = np.maximum(0, box[2:] - box[:2]).prod()
    areas = np.maximum(0, boxes[:, 2:] - boxes[:, :2]).prod(axis=1)
    return intersection / np.maximum(area + areas - intersection, 1e-12)

def average_precision_50(records, class_ids):
    """101-recall-point interpolated AP, IoU >= .50, one-to-one per-image/class.

    Records contain gt_boxes, gt_labels, pred_boxes, pred_labels, pred_scores.
    No crowd/ignore flags or maximum-detection truncation. Absent-GT classes
    have NaN AP and are excluded from mAP. A class with GT but no predictions
    has AP zero. Call after low-confidence-threshold NMS, not display filtering.
    """
    rows = []
    for cls in class_ids:
        ground_truth = [np.asarray(r['gt_boxes']).reshape(-1, 4)[np.asarray(r['gt_labels']) == cls] for r in records]
        matched = [np.zeros(len(boxes), dtype=bool) for boxes in ground_truth]
        n_gt = sum(map(len, ground_truth))
        predictions = []
        for image_index, record in enumerate(records):
            for box, label, score in zip(record['pred_boxes'], record['pred_labels'], record['pred_scores']):
                if label == cls:
                    predictions.append((float(score), image_index, np.asarray(box)))
        predictions.sort(key=lambda row: -row[0])
        tp = np.zeros(len(predictions), dtype=np.float64)
        for i, (_, image_index, box) in enumerate(predictions):
            available = np.flatnonzero(~matched[image_index])
            if available.size:
                ious = box_iou(box, ground_truth[image_index][available])
                best = int(np.argmax(ious))
                if ious[best] >= .5:
                    tp[i] = 1
                    matched[image_index][available[best]] = True
        if n_gt == 0:
            ap = float('nan')
        elif not len(predictions):
            ap = 0.0
        else:
            recall = tp.cumsum() / n_gt
            precision = tp.cumsum() / np.arange(1, len(tp) + 1)
            envelope = np.maximum.accumulate(precision[::-1])[::-1]
            ap = float(np.mean([
                envelope[np.searchsorted(recall, threshold, side='left')]
                if np.searchsorted(recall, threshold, side='left') < len(envelope) else 0.0
                for threshold in np.linspace(0, 1, 101)
            ]))
        rows.append({'class_id': cls, 'ap50': ap, 'gt_count': n_gt,
                     'ap_prediction_count': len(predictions), 'tp50': int(tp.sum()),
                     'fp50': len(predictions) - int(tp.sum())})
    values = [row['ap50'] for row in rows if np.isfinite(row['ap50'])]
    return rows, float(np.mean(values)) if values else float('nan')


In [ ]:
class SelectedPatchDataset:
    """Load the manifest's 50 already-cropped images and unchanged pixel-space GT."""
    def __init__(self, root):
        self.root = Path(root)
        self.manifest = pd.read_csv(self.root/'export_manifest.csv')
        if len(self.manifest) != 50 or self.manifest.file.nunique() != 50:
            raise ValueError('Expected exactly 50 unique exported patches')
        self.name_to_id = {name.casefold(): cls for cls, name in CLASS_NAMES.items()}
        for row in self.manifest.itertuples():
            for relative in [row.file, row.label]:
                if not (self.root/relative).is_file():
                    raise FileNotFoundError(self.root/relative)

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, index):
        row = self.manifest.iloc[index]
        with Image.open(self.root/row['file']) as image:
            if image.size != (512, 512):
                raise ValueError(f"Expected 512×512 without resizing: {row['file']}")
            pixels = np.asarray(image.convert('RGB')).copy()
        annotations = json.loads((self.root/row['label']).read_text())
        labels, boxes = [], []
        for cell in annotations['cells']:
            name = str(cell['class_name']).strip().casefold()
            if name not in self.name_to_id:
                raise ValueError(f'Unexpected exported class: {name}')
            points = np.asarray(cell['local_coordinates'], dtype=np.float64).reshape(-1, 2)
            boxes.append([*points.min(axis=0), *points.max(axis=0)])
            labels.append(self.name_to_id[name])
        boxes, valid = clip_boxes(boxes, 512, 512)
        if not valid.all():
            raise ValueError('Invalid box in exported GT')
        if len(labels) != int(row['gt_count']):
            raise ValueError('Exported GT count differs from manifest')
        return torch.from_numpy(np.ascontiguousarray(pixels.transpose(2, 0, 1))), boxes, np.asarray(labels, dtype=int)


In [ ]:
def predict_patches(dataset, checkpoint_path, device, ap_confidence=0.001, nms_iou=0.45):
    """Infer the 50 exported patches without modifying image size or GT."""
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False,
                            mmap=zipfile.is_zipfile(checkpoint_path))
    if checkpoint.get('num_classes') != 5 or checkpoint.get('architecture') != 'pdl1_flat_5class':
        raise ValueError('Expected a compatible 5-class detection checkpoint')
    names = checkpoint.get('class_names', checkpoint.get('params', {}).get('names'))
    if names is None or {int(k): v for k, v in names.items()} != CLASS_NAMES:
        raise ValueError('Checkpoint class order differs')
    with torch.no_grad():
        model = nn.yolo_v11_m(num_classes=5)
    model.load_state_dict(checkpoint['model_state_dict'], strict=True)
    del checkpoint
    model = model.to(device).eval()
    results = []  # 50개 패치의 박스/점수만 보관; 이미지와 파일은 저장하지 않음
    try:
        with torch.inference_mode():
            for index in tqdm(range(len(dataset)), desc=Path(checkpoint_path).parent.name):
                image, gt, labels = dataset[index]
                with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
                    raw = model(image.unsqueeze(0).to(device).float()/255)
                pred = util.non_max_suppression(raw.float(), confidence_threshold=ap_confidence,
                    iou_threshold=nms_iou, class_agnostic=True)[0].cpu().numpy()
                pred_boxes, valid = clip_boxes(pred[:, :4], 512, 512)
                source = str(dataset.root/dataset.manifest.iloc[index]['file'])
                results.append({'patch': source, 'source_image': source, 'gt_boxes': gt,
                    'gt_labels': labels, 'pred_boxes': pred_boxes[valid],
                    'pred_labels': pred[valid, 5].astype(int), 'pred_scores': pred[valid, 4]})
                del raw, pred, pred_boxes, image, labels, gt
    finally:
        del model
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    return results


In [ ]:
def calculate_dice(records, confidence=0.1):
    """Return union Dice, patch-level scores, and per-class pooled Dice."""
    rows = []
    for r in records:
        selected = np.asarray(r['pred_scores']) > confidence
        pred_boxes = np.asarray(r['pred_boxes'])[selected]
        pred_labels = np.asarray(r['pred_labels'])[selected]
        gt_boxes, gt_labels = np.asarray(r['gt_boxes']), np.asarray(r['gt_labels'])
        for class_id, class_name in [(-1, 'ALL')] + list(CLASS_NAMES.items()):
            gt = gt_boxes if class_id == -1 else gt_boxes[gt_labels == class_id]
            pred = pred_boxes if class_id == -1 else pred_boxes[pred_labels == class_id]
            rows.append({'patch': r['patch'], 'class_id': class_id, 'class_name': class_name,
                **rectangle_dice(gt, pred, 512, 512)})
    table = pd.DataFrame(rows)
    pooled_rows = []
    for (class_id, class_name), group in table.groupby(['class_id', 'class_name'], sort=False):
        denominator = group.gt_area.sum() + group.pred_area.sum()
        pooled_rows.append({'class_id': class_id, 'class_name': class_name,
            'dice': 2*group.intersection.sum()/denominator if denominator else float('nan')})
    pooled = pd.DataFrame(pooled_rows)
    union_dice = float(pooled.loc[pooled.class_id == -1, 'dice'].iloc[0])
    return union_dice, table, pooled[pooled.class_id != -1].reset_index(drop=True)

def calculate_map(records):
    """Class-aware mAP@0.5 over all five classes, including other."""
    rows, score = average_precision_50(records, CLASS_NAMES)
    table = pd.DataFrame(rows)
    table['class_name'] = table.class_id.map(CLASS_NAMES)
    return score, table


In [ ]:
dataset = SelectedPatchDataset(DATASET_DIR)
if not CHECKPOINT.is_file():
    raise FileNotFoundError(CHECKPOINT)
print(f'{len(dataset)} original 512×512 patches | {DEVICE}')
predictions = predict_patches(dataset, CHECKPOINT, DEVICE, AP_CONFIDENCE, NMS_IOU)


In [ ]:
dice_union, dice_table, dice_per_class = calculate_dice(predictions, CONFIDENCE)
map50, ap_per_class = calculate_map(predictions)
patch_dice = dice_table[dice_table.class_id == -1][['patch', 'dice']].reset_index(drop=True)
summary = pd.DataFrame([{'patches': len(predictions),
    'dice_mean_patch': patch_dice.dice.mean(), 'dice_union': dice_union, 'map50': map50}])
per_class = dice_per_class.merge(ap_per_class, on=['class_id', 'class_name'])
print('전체 통합 지표')
display(summary.rename(columns={'patches':'패치 수', 'dice_mean_patch':'패치별 Dice 평균',
    'dice_union':'전체 영역 통합 Dice', 'map50':'통합 mAP@0.5'}).round(6))
print('패치별 Dice')
display(patch_dice.round(6))
print('클래스별 지표')
display(per_class.round(6))
reference_path = DATASET_DIR/'selected_summary.csv'
if reference_path.is_file():
    reference = pd.read_csv(reference_path).iloc[0]
    comparison = pd.DataFrame({'metric': ['dice_mean_patch', 'dice_union', 'map50']})
    comparison['reference'] = comparison.metric.map(reference)
    comparison['current'] = comparison.metric.map(summary.iloc[0])
    comparison['difference'] = comparison['current'] - comparison['reference']
    print('저장된 기준 결과와 비교 (실제 재추론 결과)')
    display(comparison.round(8))

# 지표 계산이 끝난 예측 배열은 메모리에서 해제합니다.
del predictions
gc.collect()
